# Прогнозирование оттока клиентов банка

Учебный проект на основе предоставленного набора данных. Результаты относятся к исследованному набору и выбранному разбиению; для переноса на новые данные нужна дополнительная проверка.

In [ ]:
from pathlib import Path
DATA_PATH = Path('data/bank_clients.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Поместите исходный CSV в {DATA_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(DATA_PATH)
assert 'Exited' in df.columns, 'В CSV отсутствует целевая переменная Exited'
print('Данные загружены:', df.shape)

## Исследование данных

### Качество данных

In [ ]:
print('Размер:', df.shape)
display(df.head())
display(df.sample(5, random_state=42))
df.info()
display(df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Geography']).describe(include='all').T)


In [ ]:
print('Пропуски по столбцам:')
print(df.isna().sum().to_string())
print('Полные дубликаты:', df.duplicated().sum())
print('Уникальные значения идентификаторов:')
print(df[['RowNumber', 'CustomerId', 'Surname']].nunique().to_string())


In [ ]:
continuous = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']
q1 = df[continuous].quantile(0.25)
q3 = df[continuous].quantile(0.75)
iqr = q3 - q1
outliers = ((df[continuous] < q1 - 1.5 * iqr) | (df[continuous] > q3 + 1.5 * iqr)).sum()
print('Количество значений за границами 1,5 IQR:')
print(outliers.to_string())


**Вывод:** В таблице 2509 строк и 14 столбцов: 9 целочисленных, 2 вещественных и 3 текстовых. Пропусков и полных дубликатов строк нет. По правилу 1,5 IQR отмечены 4 значения CreditScore, 70 Age и 33 Balance; для Tenure и EstimatedSalary таких значений нет. Это кандидаты на выбросы, а не доказанные ошибки.

Минимум EstimatedSalary равен 11,58; 2 значения меньше 100 и 17 меньше 1000. Валюта, период и природа оценки неизвестны, поэтому считать эти значения ошибками и заменять их пока нет оснований.

RowNumber и CustomerId уникальны и служат идентификаторами; Surname содержит 1330 уникальных значений и не является надёжным содержательным признаком. Geography постоянно (Germany). При подготовке признаков для модели эти четыре столбца целесообразно исключить. Исходный df сохранён без удаления столбцов и строк.

### Распределения

In [ ]:
counts = df['Exited'].value_counts().sort_index()
print(pd.DataFrame({'Количество': counts, 'Доля': counts / len(df)}).round(4).to_string())
sns.countplot(data=df, x='Exited')
plt.title('Распределение оттока клиентов')
plt.xlabel('Exited (1 — ушёл)')
plt.ylabel('Количество клиентов')
plt.show()

In [ ]:
continuous = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']
categorical = ['Gender', 'NumOfProducts', 'HasCrCard', 'IsActiveMember']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, continuous):
    sns.histplot(data=df, x=col, bins=30, ax=ax)
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, categorical):
    sns.countplot(data=df, x=col, ax=ax)
plt.tight_layout()
plt.show()

print('Квантили EstimatedSalary:')
print(df['EstimatedSalary'].quantile([0, 0.001, 0.01, 0.05, 0.25, 0.5]).to_string())
print('Меньше 100:', (df['EstimatedSalary'] < 100).sum())
print('Меньше 1000:', (df['EstimatedSalary'] < 1000).sum())

In [ ]:
sns.pairplot(df, vars=continuous, hue='Exited', diag_kind='hist', plot_kws={'alpha': 0.35, 's': 12})
plt.show()

print('Медианы числовых признаков по оттоку:')
print(df.groupby('Exited')[continuous].median().round(2).to_string())
for col in categorical:
    print('\nДоля ушедших по', col)
    print(pd.crosstab(df[col], df['Exited'], normalize='index').round(3).to_string())
    print(pd.crosstab(df[col], df['Exited']).to_string())

**Вывод:** Ушли 814 клиентов (32,44%), остались 1695 (67,56%): классы несбалансированы. Медианный возраст ушедших — 45 лет, оставшихся — 36; при этом распределения пересекаются. Медианы EstimatedSalary составляют 96 498,14 и 103 557,85 соответственно; низкие значения требуют внимания, но не доказано, что они ошибочны. В дальнейшем отношении баланса к зарплате малый знаменатель может дать очень большие значения.

Доля оттока среди неактивных клиентов — 41,1%, среди активных — 23,7%; среди женщин — 37,6%, среди мужчин — 27,8%. Для одного продукта она равна 42,8%, для двух — 12,1%, для трёх — 89,6%, для четырёх — 100%; последние две группы малы (96 и 24 клиента), поэтому их доли следует трактовать осторожно. У владельцев и невладельцев кредитных карт доли близки: 32,2% и 33,0%.

Парные графики показывают значительное перекрытие классов. Эти различия описывают имеющуюся выборку и не устанавливают причин оттока; исследование проведено до разделения на обучающую и тестовую выборки.

### Корреляции

In [ ]:
features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
corr = df[features].corr(method='pearson')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True)
plt.title('Корреляции Пирсона между признаками')
plt.tight_layout()
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(key=abs, ascending=False)
print('Наибольшие по модулю межпризнаковые корреляции:')
print(pairs.head(5).round(4).to_string())
print('\nПары с |r| > 0.7:')
print(pairs[pairs.abs() > 0.7].round(4).to_string())
print('\nКорреляции признаков с Exited:')
print(df[features].corrwith(df['Exited']).sort_values(key=abs, ascending=False).round(4).to_string())

**Вывод:** Для восьми числовых признаков на всей исходной выборке пары с |r| Пирсона > 0,7 не обнаружены. Наибольшая по модулю межпризнаковая корреляция — Tenure и IsActiveMember: r = −0,0566. По этому критерию оснований удалять один из проверенных признаков нет; исходный df не изменён.

Связи с целью рассмотрены отдельно: наиболее заметна положительная корреляция Age с Exited (r = 0,3375), затем отрицательная у IsActiveMember (r = −0,1854). Корреляция NumOfProducts с целью равна −0,0729, хотя в пункте 1.2 доли оттока различались по группам: линейный коэффициент может не отражать нелинейную зависимость.

Проверка парных линейных корреляций не исключает многомерную мультиколлинеарность и нелинейные связи. Идентификаторы и постоянный Geography исключены из матрицы; Gender ещё не закодирован и в неё не вошёл. Анализ выполнен до разделения данных, поэтому при дальнейшем выборе признаков необходимо учитывать, что вся выборка уже использовалась для EDA.

### Создание признаков

In [ ]:
denominators = df[['EstimatedSalary', 'Age']]
print('Нули в знаменателях:')
print((denominators == 0).sum().to_string())
print('Пропуски в знаменателях:')
print(denominators.isna().sum().to_string())
assert denominators.notna().all().all() and (denominators != 0).all().all(), 'Проверьте знаменатели перед делением'

df_features = df.copy()
df_features['BalanceSalaryRatio'] = df_features['Balance'] / df_features['EstimatedSalary']
df_features['TenureByAge'] = df_features['Tenure'] / df_features['Age']
df_features['CreditScoreGivenAge'] = df_features['CreditScore'] / df_features['Age']
new_features = ['BalanceSalaryRatio', 'TenureByAge', 'CreditScoreGivenAge']
print('\nПропуски и бесконечности в новых признаках:')
print(pd.DataFrame({'Пропуски': df_features[new_features].isna().sum(), 'Бесконечности': np.isinf(df_features[new_features]).sum()}).to_string())
print('\nРаспределения новых признаков:')
print(df_features[new_features].describe(percentiles=[0.5, 0.95, 0.99, 0.999]).round(4).to_string())
print('\nПять наибольших отношений баланса к зарплате:')
print(df_features.nlargest(5, 'BalanceSalaryRatio')[['Balance', 'EstimatedSalary', 'BalanceSalaryRatio']].round(4).to_string(index=False))

**Вывод:** Три признака созданы в копии `df_features`; исходный `df` сохранён. В `Age` и `EstimatedSalary` нулей и пропусков нет; в новых признаках нет пропусков и бесконечных значений.

`BalanceSalaryRatio` имеет длинный правый хвост: медиана 1,1972, 99-й процентиль 65,0159, максимум 10 614,6554. Максимум получен при балансе 122 917,71 и предполагаемой зарплате 11,58: малый знаменатель резко увеличивает отношение. Ошибочность исходной оценки зарплаты не подтверждена, поэтому значения оставлены. При моделировании следует учитывать влияние экстремальных значений; решение об обработке нужно проверять на обучающей выборке.

`TenureByAge` находится в диапазоне от 0 до 0,5, `CreditScoreGivenAge` — от 6,1127 до 46,3889.

In [ ]:
columns = ['EstimatedSalary', 'BalanceSalaryRatio']
limits = df_features[columns].quantile([0.01, 0.99])
flags = pd.DataFrame({
    'salary_low': df_features['EstimatedSalary'] <= limits.loc[0.01, 'EstimatedSalary'],
    'salary_high': df_features['EstimatedSalary'] >= limits.loc[0.99, 'EstimatedSalary'],
    'ratio_low': df_features['BalanceSalaryRatio'] <= limits.loc[0.01, 'BalanceSalaryRatio'],
    'ratio_high': df_features['BalanceSalaryRatio'] >= limits.loc[0.99, 'BalanceSalaryRatio']
})
selected = flags.any(axis=1)
diagnostic = df_features.loc[selected, ['EstimatedSalary', 'Balance', 'BalanceSalaryRatio']].join(flags.loc[selected])
diagnostic.index.name = 'Индекс df'
diagnostic = diagnostic.sort_values('EstimatedSalary')
print('Пороги отбора (включая границы):')
print(limits.to_string())
print('\nКоличество по условиям:')
print(flags.sum().to_string())
print('Уникальных строк:', len(diagnostic))
with pd.option_context('display.max_rows', None):
    display(diagnostic)

**Условное допущение для диагностики:** считаем `EstimatedSalary` месячной зарплатой в евро до налогов при полной занятости. Для сравнения используем ориентир Германии за 2026 год — 2343 евро в месяц, рассчитанный Destatis на основе ставки 13,90 евро в час ([источник](https://www.destatis.de/Europa/EN/Topic/Population-Labour-Social-Issues/Labour-market/MinimumWages.html?templateQueryString=wages)).

В описании датасета валюта, период, год, занятость и метод оценки зарплаты не указаны. Поэтому применимость этого ориентира к данным не подтверждена: значения ниже него отмечаются для проверки и не считаются доказанными ошибками.

In [ ]:
salary_review = df_features.loc[df_features['EstimatedSalary'] < 2343, ['EstimatedSalary', 'Balance', 'BalanceSalaryRatio']].sort_values('EstimatedSalary')
salary_review.index.name = 'Индекс df'
print('Значений ниже условного ориентира 2343:', len(salary_review))
with pd.option_context('display.max_rows', None):
    display(salary_review)

### Кодирование категорий

In [ ]:
gender_map = {'Female': 0, 'Male': 1}
unknown_gender = df_features['Gender'].dropna()[~df_features['Gender'].dropna().isin(gender_map)]
print('Неизвестные категории Gender:', unknown_gender.unique().tolist())
print('Пропуски Gender:', df_features['Gender'].isna().sum())
assert unknown_gender.empty and df_features['Gender'].notna().all(), 'Проверьте значения Gender'

model_df = df_features.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Geography']).copy()
model_df['Gender'] = model_df['Gender'].map(gender_map)
print('Размер model_df:', model_df.shape)
print('Gender после кодирования:')
print(model_df['Gender'].value_counts().sort_index().to_string())
print('Пропуски:', model_df.isna().sum().sum())
print('Нечисловые столбцы:', model_df.select_dtypes(exclude='number').columns.tolist())
print('Цель Exited:', model_df['Exited'].value_counts().sort_index().to_dict())
display(model_df.head())

**Вывод:** `Gender` — бинарный категориальный признак, поэтому закодирован одним столбцом: Female = 0, Male = 1. Неизвестных категорий и пропусков в нём нет. `HasCrCard` и `IsActiveMember` уже заданы как 0/1, а `Age` и `NumOfProducts` являются числовыми признаками и не требуют такого кодирования.

Из `model_df` исключены идентификаторы `RowNumber`, `CustomerId`, фамилия `Surname` и постоянный в этой выборке `Geography`. Получена числовая таблица 2509 × 13 без пропусков; `Exited` сохранён как целевая переменная и при последующем разделении будет отделён от признаков. Исходные `df` и `df_features` не изменены.

### Разделение выборки

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns='Exited')
y = model_df['Exited']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Размеры train и test:', X_train.shape, X_test.shape)
print('Пересечение индексов:', len(X_train.index.intersection(X_test.index)))
for name, target in [('train', y_train), ('test', y_test)]:
    print(f'\n{name}:')
    print(pd.DataFrame({'Количество': target.value_counts().sort_index(), 'Доля': target.value_counts(normalize=True).sort_index()}).round(4).to_string())

### Масштабирование

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

binary = ['Gender', 'HasCrCard', 'IsActiveMember']
ratio = ['BalanceSalaryRatio']
continuous = [col for col in X_train.columns if col not in binary + ratio]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
preprocessors = {}
for scaler_name, scaler in [('Standard', StandardScaler()), ('Robust', RobustScaler())]:
    for use_log in [False, True]:
        name = scaler_name + (' + log1p' if use_log else '')
        ratio_steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if use_log else []
        ratio_steps.append(('scale', clone(scaler)))
        prep = ColumnTransformer([
            ('continuous', clone(scaler), continuous),
            ('ratio', Pipeline(ratio_steps), ratio),
            ('binary', 'passthrough', binary)
        ]).set_output(transform='pandas')
        model = Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=2000, random_state=42))])
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
        preprocessors[name] = prep
        results.append({'Вариант': name, 'ROC-AUC среднее': scores.mean(), 'Стандартное отклонение': scores.std(), 'Фолды': np.round(scores, 4).tolist()})

comparison = pd.DataFrame(results).set_index('Вариант').sort_values('ROC-AUC среднее', ascending=False)
display(comparison)
best_name = comparison.index[0]
preprocessor = clone(preprocessors[best_name])
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)
print('Выбранный вариант:', best_name)
print('Размеры после преобразования:', X_train_scaled.shape, X_test_scaled.shape)
print('Бинарные значения train/test:', {c: sorted(X_train_scaled[c].unique().tolist()) for c in X_train_scaled if c.startswith('binary__')}, {c: sorted(X_test_scaled[c].unique().tolist()) for c in X_test_scaled if c.startswith('binary__')})

**Вывод:** Сравнены StandardScaler и RobustScaler с `log1p` отношения `BalanceSalaryRatio` и без него. Для всех вариантов использована одна логистическая регрессия и стратифицированная 5-fold кросс-валидация на обучающей выборке; преобразования обучались внутри каждого фолда. Двоичные признаки сохранены как 0/1.

Предварительная метрика ROC-AUC выбрана для сравнения ранжирования без выбора порога: бизнес-задача учитывает и ложноположительные, и ложноотрицательные ошибки, поэтому целевую пороговую метрику рассмотрим при выборе модели. Наибольшее среднее ROC-AUC получено для StandardScaler + log1p: 0,7618 (стандартное отклонение по фолдам 0,0217). Остальные варианты дают 0,7599–0,7617; различия малы относительно разброса и не доказывают устойчивого преимущества. `log1p` сжимает длинный правый хвост отношения, но не исправляет исходные значения.

По объявленному критерию выбран StandardScaler + log1p. Преобразователь обучен только на всём train и применён к train и test.

## Базовая модель

### Метрики

**Ответ:** Основная метрика — F1 для класса `Exited = 1`. Она объединяет precision и recall: precision показывает, какая доля клиентов, которым модель предложила удержание, действительно относится к классу оттока, а recall — какую долю ушедших клиентов удалось обнаружить. Высокая precision помогает сократить лишние звонки, а высокий recall — не пропустить уходящих клиентов.

F1 придаёт precision и recall одинаковый вес; выбираем это допущение, поскольку стоимость разных ошибок в условии не задана. Дополнительно будем смотреть precision и recall отдельно и при необходимости матрицу ошибок. ROC-AUC в пункте 1.7 использовалась как вспомогательная метрика, не зависящая от порога.

### Логистическая регрессия

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

logreg = LogisticRegression(max_iter=2000, random_state=42)
logreg.fit(X_train_scaled, y_train)

metrics = []
for name, X_part, y_part in [('train', X_train_scaled, y_train), ('test', X_test_scaled, y_test)]:
    predictions = logreg.predict(X_part)
    probabilities = logreg.predict_proba(X_part)[:, 1]
    metrics.append({'Выборка': name, 'F1': f1_score(y_part, predictions),
                    'Precision': precision_score(y_part, predictions),
                    'Recall': recall_score(y_part, predictions),
                    'ROC-AUC': roc_auc_score(y_part, probabilities)})
    print(f'Матрица ошибок {name} (строки: истинный класс, столбцы: предсказанный):')
    print(confusion_matrix(y_part, predictions))
display(pd.DataFrame(metrics).set_index('Выборка').round(4))

coefficients = pd.Series(logreg.coef_[0], index=X_train_scaled.columns, name='Коэффициент')
display(coefficients.sort_values(key=abs, ascending=False).to_frame().round(4))

In [ ]:
metric_table = pd.DataFrame(metrics).set_index('Выборка')[['F1', 'Precision', 'Recall', 'ROC-AUC']]
ax = metric_table.T.plot(kind='bar', figsize=(8, 4), color=['#4c78a8', '#f58518'], rot=0)
ax.set(title='Качество логистической регрессии', xlabel='Метрика', ylabel='Значение', ylim=(0, 1))
ax.legend(['Обучающая', 'Тестовая'], title='Выборка')
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

test_probabilities = logreg.predict_proba(X_test_scaled)[:, 1]
test_predictions = logreg.predict(X_test_scaled)
precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, test_probabilities)
average_precision = average_precision_score(y_test, test_probabilities)
base_rate = y_test.mean()

plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f'PR-кривая (AP = {average_precision:.3f})')
plt.axhline(base_rate, color='gray', linestyle='--', label=f'Доля оттока = {base_rate:.3f}')
plt.scatter(recall_score(y_test, test_predictions), precision_score(y_test, test_predictions),
            color='red', zorder=3, label='Порог 0,5')
plt.xlabel('Recall (полнота)')
plt.ylabel('Precision (точность)')
plt.title('Precision–Recall на тестовой выборке')
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f'Average Precision: {average_precision:.4f}; доля оттока: {base_rate:.4f}')

**Вывод:** Логистическая регрессия обучена на train после выбранной подготовки признаков; для классов использован стандартный порог 0,5. На train F1 = 0,5352, precision = 0,6383, recall = 0,4608, ROC-AUC = 0,7695; на test — соответственно 0,4509, 0,5536, 0,3804 и 0,7069. На test модель выявила 62 из 163 ушедших клиентов, пропустила 101 и ошибочно отнесла к оттоку 50 оставшихся.

Метрики на test ниже train; по одному разбиению нельзя уверенно установить причину или степень переобучения. Порог по test не подбирался.

Наибольшие по модулю коэффициенты получены для `CreditScoreGivenAge` (−1,0994) и `IsActiveMember` (−0,9961), далее `CreditScore` (0,4509) и `Gender` (−0,3900). Отрицательный коэффициент соответствует уменьшению логарифма шансов оттока при прочих равных. Непрерывные признаки стандартизированы, а бинарные оставлены 0/1, поэтому их коэффициенты относятся к разным изменениям признаков; связанные исходные и производные признаки также ограничивают интерпретацию. Коэффициенты описывают модель, а не причинное влияние.

## Нелинейные модели

### Полиномиальная логистическая регрессия

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_model = Pipeline([
    ('preprocess', clone(preprocessors[best_name])),
    ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])
poly_model.fit(X_train, y_train)

poly_metrics = []
for name, X_part, y_part in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predicted = poly_model.predict(X_part)
    probability = poly_model.predict_proba(X_part)[:, 1]
    poly_metrics.append({'Выборка': name, 'F1': f1_score(y_part, predicted),
                         'Precision': precision_score(y_part, predicted),
                         'Recall': recall_score(y_part, predicted),
                         'ROC-AUC': roc_auc_score(y_part, probability)})

comparison_models = pd.concat({'Baseline': pd.DataFrame(metrics).set_index('Выборка'),
                               'Полином 2': pd.DataFrame(poly_metrics).set_index('Выборка')}, names=['Модель'])
display(comparison_models.round(4))

feature_names = poly_model.named_steps['polynomial'].get_feature_names_out(
    poly_model.named_steps['preprocess'].get_feature_names_out())
poly_coefficients = pd.Series(poly_model.named_steps['model'].coef_[0], index=feature_names, name='Коэффициент')
print('Количество полиномиальных признаков:', len(feature_names))
print('Итераций:', poly_model.named_steps['model'].n_iter_[0], 'из', poly_model.named_steps['model'].max_iter)
display(poly_coefficients.sort_values(key=abs, ascending=False).to_frame().round(4))

**Вывод:** Применены выбранная подготовка признаков, полиномиальные признаки второй степени и повторное масштабирование; оба масштабирования обучены только на train. Получено 90 признаков. Логистическая регрессия с регуляризацией по умолчанию завершила оптимизацию за 68 итераций из 2000 без предупреждения о сходимости.

При пороге 0,5 полиномиальная модель дала на train F1 = 0,7098, precision = 0,7712, recall = 0,6575, ROC-AUC = 0,8923; на test — 0,6340, 0,6783, 0,5951 и 0,8309. Для baseline F1 на test = 0,4509, ROC-AUC = 0,7069. Таким образом, на данном разделении показатели улучшились. Результат на train выше, чем на test, поэтому разрыв обобщения сохраняется. Порог и параметры по test не подбирались.

Наибольшие по модулю веса имеют квадрат `NumOfProducts` (2,3435), взаимодействие `Age` и `CreditScoreGivenAge` (−1,4036), квадрат `Age` (−1,3831) и взаимодействие `CreditScoreGivenAge` с `IsActiveMember` (−1,2131). Коэффициенты относятся к стандартизированным полиномиальным признакам после исходной подготовки; их нельзя напрямую трактовать как влияние исходного признака отдельно. Взаимосвязь признаков, включая повторение квадратов бинарных 0/1, ограничивает устойчивость отдельных весов. Полученные различия описывают это разделение, а не гарантируют такого же преимущества на новых данных.

### Метод опорных векторов

In [ ]:
from sklearn.svm import SVC

kernel_results = []
for kernel in ['linear', 'rbf', 'poly', 'sigmoid']:
    candidate = Pipeline([('preprocess', clone(preprocessors[best_name])),
                          ('model', SVC(kernel=kernel, random_state=42))])
    scores = cross_val_score(candidate, X_train, y_train, cv=cv, scoring='f1')
    kernel_results.append({'Ядро': kernel, 'F1 среднее': scores.mean(),
                           'Стандартное отклонение': scores.std(), 'Фолды': np.round(scores, 4).tolist()})
kernel_comparison = pd.DataFrame(kernel_results).set_index('Ядро').sort_values('F1 среднее', ascending=False)
display(kernel_comparison)
best_kernel = kernel_comparison.index[0]
print('Выбрано по среднему F1 на CV:', best_kernel)

In [ ]:
svm_model = Pipeline([('preprocess', clone(preprocessors[best_name])),
                      ('model', SVC(kernel=best_kernel, random_state=42))])
svm_model.fit(X_train, y_train)

svm_metrics = []
for name, X_part, y_part in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predicted = svm_model.predict(X_part)
    scores = svm_model.decision_function(X_part)
    svm_metrics.append({'Выборка': name, 'F1': f1_score(y_part, predicted),
                        'Precision': precision_score(y_part, predicted),
                        'Recall': recall_score(y_part, predicted),
                        'ROC-AUC': roc_auc_score(y_part, scores)})
comparison_svm = pd.concat({'Baseline': pd.DataFrame(metrics).set_index('Выборка'),
                            'Полином 2': pd.DataFrame(poly_metrics).set_index('Выборка'),
                            'SVM': pd.DataFrame(svm_metrics).set_index('Выборка')}, names=['Модель'])
display(comparison_svm.round(4))
print('Итераций SVM:', svm_model.named_steps['model'].n_iter_[0],
      '; статус fit (0 — сходимость):', svm_model.named_steps['model'].fit_status_)

In [ ]:
if best_kernel == 'linear':
    linear_svm = svm_model
else:
    print(f'У выбранного ядра {best_kernel} нет явных коэффициентов исходных признаков.')
    linear_svm = Pipeline([('preprocess', clone(preprocessors[best_name])),
                           ('model', SVC(kernel='linear', random_state=42))])
    linear_svm.fit(X_train, y_train)
    print('Коэффициенты ниже принадлежат отдельному линейному SVM, не выбранной модели:')

linear_coefficients = pd.Series(linear_svm.named_steps['model'].coef_[0],
    index=linear_svm.named_steps['preprocess'].get_feature_names_out(), name='Коэффициент')
display(linear_coefficients.sort_values(key=abs, ascending=False).to_frame().round(4))

**Вывод:** Ядра SVM сравнивались по F1 на стратифицированной 5-fold кросс-валидации обучающей выборки; подготовка обучалась внутри каждого фолда. Средний F1: RBF — 0,6723 (стандартное отклонение 0,0097), poly — 0,5714, linear — 0,5043, sigmoid — 0,4177. По этому критерию выбрано RBF и обучено на всём train.

На train F1 = 0,7218, precision = 0,8216, recall = 0,6436, ROC-AUC = 0,8916; на test — 0,6224, 0,7236, 0,5460 и 0,8169. ROC-AUC рассчитана по `decision_function`, а не по вероятностям. На данном разделении F1 на test выше baseline (0,4509) и немного ниже полиномиальной логистической регрессии (0,6340); разница сама по себе не доказывает устойчивого превосходства. Результат на train выше test. Ядро и порог по test не подбирались.

У выбранного RBF нет явного вектора коэффициентов исходных признаков. Для иллюстрации отдельно обучен линейный SVM на train: его крупнейшие по модулю веса — `Age` (0,8830) и `IsActiveMember` (−0,6100). Они относятся к линейной альтернативе, а не к выбранному RBF; различие масштабов бинарных и непрерывных признаков, а также зависимости между признаками ограничивают интерпретацию весов.

### Подбор порога на обучающих данных

In [ ]:
from sklearn.model_selection import cross_val_predict

oof_prob = cross_val_predict(clone(poly_model), X_train, y_train, cv=cv, method='predict_proba')[:, 1]
precisions, recalls, candidates = precision_recall_curve(y_train, oof_prob)
precisions, recalls = precisions[:-1], recalls[:-1]
f1_values = np.divide(2 * precisions * recalls, precisions + recalls,
                      out=np.zeros_like(precisions), where=(precisions + recalls) != 0)
best_index = np.flatnonzero(f1_values == f1_values.max())[-1]
best_threshold = candidates[best_index]
print(f'Порог: {best_threshold:.6f}; F1 OOF: {f1_values[best_index]:.4f}; '
      f'precision: {precisions[best_index]:.4f}; recall: {recalls[best_index]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(candidates, f1_values, label='F1')
plt.plot(candidates, precisions, label='Precision', alpha=0.8)
plt.plot(candidates, recalls, label='Recall', alpha=0.8)
plt.axvline(best_threshold, color='black', linestyle='--', label=f'Выбранный порог = {best_threshold:.3f}')
plt.xlabel('Порог вероятности')
plt.ylabel('Значение метрики')
plt.title('Выбор порога по OOF-предсказаниям train')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
poly_test_prob = poly_model.predict_proba(X_test)[:, 1]
test_results = []
for label, prediction in [('По умолчанию (0,5)', poly_model.predict(X_test)),
                          ('Выбранный на train', (poly_test_prob >= best_threshold).astype(int))]:
    test_results.append({'Порог': label, 'F1': f1_score(y_test, prediction),
                         'Precision': precision_score(y_test, prediction),
                         'Recall': recall_score(y_test, prediction),
                         'ROC-AUC': roc_auc_score(y_test, poly_test_prob)})
    print(f'Матрица ошибок: {label} (строки — факт, столбцы — прогноз)')
    print(confusion_matrix(y_test, prediction))
display(pd.DataFrame(test_results).set_index('Порог').round(4))

**Вывод:** Для полиномиальной логистической регрессии порог выбран по out-of-fold вероятностям на train при стратифицированной 5-fold кросс-валидации: все преобразования обучались внутри каждого фолда. Рассмотрены все уникальные вероятности как кандидаты; при равном максимальном F1 выбирается наибольший порог. Выбран порог 0,344667, при котором F1 на OOF = 0,7023, precision = 0,6481, recall = 0,7665. Правило для выбранного порога: вероятность ≥ порога. Значение F1 на тех же OOF-данных, где искался максимум, не является независимой оценкой качества выбранного порога.

На test при стандартном пороге 0,5 F1 = 0,6340, precision = 0,6783, recall = 0,5951. При пороге, выбранном на train, F1 = 0,6592, precision = 0,6094, recall = 0,7178: найдено 117 вместо 97 ушедших клиентов, но ошибочных срабатываний стало 75 вместо 46. ROC-AUC остаётся 0,8309, поскольку вероятности модели не изменились.

Порог выбирался по обучающим данным. Результат относится к этому разбиению и не гарантирует улучшения на новых данных.

### Дополнительный эксперимент: сравнение моделей

Ниже приведено сравнение полиномиальной логистической регрессии и SVM RBF с подбором порогов на обучающих данных.

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=43)
assert best_kernel == 'rbf'
models = {'Полиномиальная логистическая регрессия': (poly_model, 'predict_proba'),
          'SVM RBF': (svm_model, 'decision_function')}
fold_results = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_train, y_train), 1):
    X_inner, y_inner = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    for name, (pipeline, method) in models.items():
        inner_scores = cross_val_predict(clone(pipeline), X_inner, y_inner, cv=inner_cv, method=method)
        if method == 'predict_proba':
            inner_scores = inner_scores[:, 1]
        p, r, candidates = precision_recall_curve(y_inner, inner_scores)
        f1 = np.divide(2 * p[:-1] * r[:-1], p[:-1] + r[:-1],
                       out=np.zeros_like(candidates), where=(p[:-1] + r[:-1]) != 0)
        threshold = candidates[np.flatnonzero(f1 == f1.max())[-1]]
        fitted = clone(pipeline).fit(X_inner, y_inner)
        val_scores = fitted.predict_proba(X_val)[:, 1] if method == 'predict_proba' else fitted.decision_function(X_val)
        fold_results.append({'Фолд': fold, 'Модель': name, 'Порог score': threshold,
                             'F1': f1_score(y_val, val_scores >= threshold)})

fold_table = pd.DataFrame(fold_results)
display(fold_table.round(4))
summary = fold_table.groupby('Модель')['F1'].agg(['mean', lambda values: values.std(ddof=0)])
summary.columns = ['Средний F1', 'SD по фолдам']
display(summary.round(4))

**Дополнительный эксперимент.** Модели сравниваются на одних и тех же пяти внешних стратифицированных фолдах train. На каждом внешнем обучающем наборе порог для каждой модели выбирается по максимуму F1 на четырёх внутренних OOF-фолдах; при равенстве выбирается наибольший порог. Преобразования заново обучаются внутри каждого фолда. Для логистической регрессии порог применяется к вероятности, для SVM — к значению decision_function, которое не является вероятностью.

Средний F1 полиномиальной логистической регрессии — 0,6953 (SD 0,0155), SVM RBF — 0,6954 (SD 0,0134). Разница средних на этих разбиениях мала относительно разброса по фолдам и не доказывает превосходства одной модели. Test в эксперименте не используется. Ограничение: вариант preprocessing и ядро RBF ранее выбраны на всей train-выборке и здесь зафиксированы; поэтому это не полностью вложенная оценка всей процедуры их выбора.

In [ ]:
test_comparison = []
for name, (pipeline, method) in models.items():
    oof_scores = cross_val_predict(clone(pipeline), X_train, y_train, cv=cv, method=method)
    if method == 'predict_proba':
        oof_scores = oof_scores[:, 1]
    precision_oof, recall_oof, score_thresholds = precision_recall_curve(y_train, oof_scores)
    denom = precision_oof[:-1] + recall_oof[:-1]
    f1_oof = np.divide(2 * precision_oof[:-1] * recall_oof[:-1], denom,
                       out=np.zeros_like(score_thresholds), where=denom != 0)
    selected_threshold = score_thresholds[np.flatnonzero(f1_oof == f1_oof.max())[-1]]
    fitted_model = clone(pipeline).fit(X_train, y_train)
    scores_test = fitted_model.predict_proba(X_test)[:, 1] if method == 'predict_proba' else fitted_model.decision_function(X_test)
    predictions_test = scores_test >= selected_threshold
    test_comparison.append({'Модель': name, 'Порог score': selected_threshold,
                            'F1': f1_score(y_test, predictions_test),
                            'Precision': precision_score(y_test, predictions_test),
                            'Recall': recall_score(y_test, predictions_test),
                            'ROC-AUC': roc_auc_score(y_test, scores_test)})
    print(name, '— матрица ошибок (строки: факт, столбцы: прогноз):')
    print(confusion_matrix(y_test, predictions_test))

display(pd.DataFrame(test_comparison).set_index('Модель').round(4))

**Сравнение на test.** Для каждой фиксированной модели порог выбран по максимуму F1 на OOF-прогнозах train с одинаковыми пятью стратифицированными фолдами; при равенстве выбран наибольший порог. После этого pipeline обучен на всём train и оценён на test. Для SVM порог относится к decision_function и не является порогом вероятности.

На этом разбиении полиномиальная логистическая регрессия получила F1 = 0,6592, precision = 0,6094 и recall = 0,7178; SVM RBF — F1 = 0,6341, precision = 0,6303 и recall = 0,6380. Первая модель обнаружила больше ушедших клиентов, но допустила больше ложных срабатываний. Сравнение на одном разбиении не доказывает устойчивого превосходства.